# sbic-tracker — SBIC Investment Portfolio Analyzer
## Demo: Analyzing Small Business Investment Company Portfolios

This notebook demonstrates how to use sbic-tracker to:
- Load SBIC licensee and investment data
- Compute fund-level performance metrics (IRR, TVPI, DPI, RVPI)
- Analyze vintage year cohorts and peer quartile rankings
- Track investments by NAICS sector and geography
- Build a comprehensive portfolio tracker with summary statistics

Background: The SBIC program is a $30B+ SBA initiative that licenses funds to deploy
leverage capital into US small businesses. 300+ active SBICs deploy $8B+ annually,
but the data lives in inconsistently formatted SBA Excel and PDF files with no API.


In [ ]:
import sys
sys.path.insert(0, '..')

from sbictracker import (
    SBICLicensee, Investment, SBICPortfolio,
    irr, tvpi, dpi, rvpi, called_capital, distributed_capital, nav,
    vintage_year_analysis, peer_quartile_ranking,
    sector_breakdown, state_breakdown, top_naics,
    load_sample_licensees, load_sample_investments,
    NAICS_SECTORS, LICENSE_TYPES, INVESTMENT_INSTRUMENTS,
)
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("sbic-tracker loaded successfully")
print(f"\nLicense types:        {list(LICENSE_TYPES.keys())}")
print(f"Investment types:     {list(INVESTMENT_INSTRUMENTS.keys())}")
print(f"NAICS sectors tracked: {len(NAICS_SECTORS)}")


## 1. Load Sample SBIC Data

We use synthetic sample data here that reflects realistic SBIC structures.
For real data, parse SBA quarterly licensee reports.


In [ ]:
licensees = load_sample_licensees()
investments = load_sample_investments()

print(f"Loaded {len(licensees)} SBIC licensees")
print(f"Loaded {len(investments)} portfolio investments\n")

print("Sample SBIC Licensees:")
print("=" * 75)
for lic in licensees[:5]:
    total_cap = lic.total_capital / 1e6
    print(f"  {lic.fund_name:<35} {lic.license_type:<15} ${total_cap:>6.0f}MM  {lic.license_status}")


## 2. Build a Portfolio Tracker

The SBICPortfolio class organizes investments and computes aggregate metrics.


In [ ]:
portfolio = SBICPortfolio(name="Sample SBIC Fund I")

for inv in investments[:20]:
    portfolio.add(inv)

print(f"Portfolio: {portfolio.name}")
print(f"Total investments:    {portfolio.count()}")
print(f"Total invested:       ${portfolio.total_invested/1e6:.1f}MM")
print(f"Total realized:       ${portfolio.total_realized/1e6:.1f}MM")
print(f"Total written off:    ${portfolio.total_written_off/1e6:.1f}MM")


## 3. Compute Fund-Level Performance Metrics

The standard PE/VC performance metrics: IRR, TVPI, DPI, and RVPI.


In [ ]:
# Compute on the underlying investments
fund_irr = irr(investments[:20])
fund_tvpi = tvpi(investments[:20])
fund_dpi = dpi(investments[:20])
fund_rvpi = rvpi(investments[:20])
called = called_capital(investments[:20])
distributed = distributed_capital(investments[:20])
fund_nav = nav(investments[:20])

print("Fund Performance Metrics:")
print("=" * 50)
print(f"  Net IRR:              {fund_irr*100:>6.2f}%")
print(f"  TVPI (Total/Paid-in): {fund_tvpi:>6.2f}x")
print(f"  DPI (Distributed):    {fund_dpi:>6.2f}x")
print(f"  RVPI (Residual):      {fund_rvpi:>6.2f}x")
print()
print(f"  Called Capital:       ${called/1e6:>6.1f}MM")
print(f"  Distributed:          ${distributed/1e6:>6.1f}MM")
print(f"  NAV (residual value): ${fund_nav/1e6:>6.1f}MM")


## 4. Vintage Year Analysis

Compare fund performance by vintage year to identify market cycles.


In [ ]:
vintage_df = vintage_year_analysis(investments)
print("Performance by Vintage Year:")
print("=" * 70)
print(vintage_df.to_string(index=False))


## 5. Peer Quartile Ranking

Rank SBICs against peer funds by IRR.


In [ ]:
ranking = peer_quartile_ranking(licensees)
print("Peer Quartile Rankings:")
print("=" * 60)
print(ranking.head(10).to_string(index=False))


## 6. Sector Breakdown by NAICS

Where is the SBIC capital deployed?


In [ ]:
sectors = sector_breakdown(investments)
print("Investment Distribution by Sector:")
print("=" * 70)
print(sectors.head(10).to_string(index=False))

print("\n\nTop NAICS Codes by Investment Count:")
top = top_naics(investments, n=10)
print(top.to_string(index=False))


## 7. Geographic Distribution

In [ ]:
states = state_breakdown(investments)
print("Investment Distribution by State:")
print("=" * 60)
print(states.head(10).to_string(index=False))


## 8. Filter and Analyze Subsets

Filter investments by sector or state to do deep dives.


In [ ]:
# Filter by state
ny_investments = portfolio.filter_state("NY")
print(f"NY investments: {len(ny_investments)}")
if ny_investments:
    print(f"NY IRR: {irr(ny_investments)*100:.2f}%")

# Filter by NAICS sector (manufacturing)
manufacturing = portfolio.filter_sector("Manufacturing")
print(f"\nManufacturing investments: {len(manufacturing)}")
if manufacturing:
    print(f"Manufacturing IRR: {irr(manufacturing)*100:.2f}%")


## 9. Active vs Wound-Down Funds

Track licensee status across the SBIC universe.


In [ ]:
active = [l for l in licensees if l.license_status == "active"]
wound_down = [l for l in licensees if l.license_status == "wound_down"]
surrendered = [l for l in licensees if l.license_status == "surrendered"]

print(f"Active SBICs:        {len(active):>3}")
print(f"Wound-down SBICs:    {len(wound_down):>3}")
print(f"Surrendered SBICs:   {len(surrendered):>3}")

active_capital = sum(l.total_capital for l in active) / 1e6
print(f"\nTotal active capital: ${active_capital:,.0f}MM")


## Summary

This notebook demonstrated the complete SBIC analysis workflow:

1. **Load licensee and investment data** — sample or live SBA reports
2. **Portfolio tracking** — aggregate metrics across investments
3. **Fund metrics** — IRR, TVPI, DPI, RVPI, called/distributed capital
4. **Vintage analysis** — compare performance across fund vintages
5. **Peer ranking** — quartile rankings against peer SBICs
6. **Sector & geography** — NAICS and state breakdowns
7. **Filtering** — analyze subsets by state, sector, or instrument type
8. **Status tracking** — active vs wound-down vs surrendered licensees

**Key use case:** SBIC LP analysts evaluating fund commitments,
SBA program staff monitoring program health, and impact investors
seeking sector-specific exposure all need this data — and currently
nothing exists in open source.

**GitHub:** https://github.com/Jaypatel1511/sbic-tracker
**PyPI:** https://pypi.org/project/sbic-tracker
**Data:** https://www.sba.gov/funding-programs/investment-capital
